In [ ]:
from google.colab import drive

drive.mount('/content/drive')

dataset_path = "/content/drive/MyDrive/dataset"

train_path = dataset_path + "/train"
validation_path = dataset_path + "/validation"
test_path = dataset_path + "/test"

Mounted at /content/drive


In [ ]:
import tensorflow as tf

train_images = tf.keras.utils.image_dataset_from_directory(
    train_path + "/images",
    labels=None,
    image_size=(128, 128),
    batch_size=32,
    shuffle=False,
    color_mode="rgb"
)

train_masks = tf.keras.utils.image_dataset_from_directory(
    train_path + "/masks",
    labels=None,
    image_size=(128, 128),
    batch_size=32,
    shuffle=False,
    color_mode="grayscale"
)


valid_images = tf.keras.utils.image_dataset_from_directory(
    validation_path + "/images",
    labels=None,
    image_size=(128, 128),
    batch_size=32,
    shuffle=False,
    color_mode="rgb"
)

valid_masks = tf.keras.utils.image_dataset_from_directory(
    validation_path + "/masks",
    labels=None,
    image_size=(128, 128),
    batch_size=32,
    shuffle=False,
    color_mode="grayscale"
)


test_images = tf.keras.utils.image_dataset_from_directory(
    test_path + "/images",
    labels=None,
    image_size=(128, 128),
    batch_size=32,
    shuffle=False,
    color_mode="rgb"
)

test_masks = tf.keras.utils.image_dataset_from_directory(
    test_path + "/masks",
    labels=None,
    image_size=(128, 128),
    batch_size=32,
    shuffle=False,
    color_mode="grayscale"
)

Found 1385 files.
Found 1385 files.
Found 396 files.
Found 396 files.
Found 199 files.
Found 199 files.


In [ ]:
train_data = tf.data.Dataset.zip(
    (train_images, train_masks)
)

valid_data = tf.data.Dataset.zip(
    (valid_images, valid_masks)
)

test_data = tf.data.Dataset.zip(
    (test_images, test_masks)
)

In [ ]:
train_data = train_data.map(
    lambda image, mask: (image / 255.0, mask / 255.0)
)

valid_data = valid_data.map(
    lambda image, mask: (image / 255.0, mask / 255.0)
)

test_data = test_data.map(
    lambda image, mask: (image / 255.0, mask / 255.0)
)

In [ ]:
train_data = train_data.shuffle(1385)

In [ ]:
from tensorflow.keras import layers, Model

inputs = layers.Input((128, 128, 3))

x1 = layers.Conv2D(32, 3, activation="relu", padding="same")(inputs)
x1 = layers.Conv2D(32, 3, activation="relu", padding="same")(x1)
p1 = layers.MaxPooling2D()(x1)

x2 = layers.Conv2D(64, 3, activation="relu", padding="same")(p1)
x2 = layers.Conv2D(64, 3, activation="relu", padding="same")(x2)
p2 = layers.MaxPooling2D()(x2)

x3 = layers.Conv2D(128, 3, activation="relu", padding="same")(p2)
x3 = layers.Conv2D(128, 3, activation="relu", padding="same")(x3)

u1 = layers.UpSampling2D()(x3)
u1 = layers.Concatenate()([u1, x2])
u1 = layers.Conv2D(64, 3, activation="relu", padding="same")(u1)
u1 = layers.Conv2D(64, 3, activation="relu", padding="same")(u1)

u2 = layers.UpSampling2D()(u1)
u2 = layers.Concatenate()([u2, x1])
u2 = layers.Conv2D(32, 3, activation="relu", padding="same")(u2)
u2 = layers.Conv2D(32, 3, activation="relu", padding="same")(u2)

outputs = layers.Conv2D(1, 1, activation="sigmoid")(u2)

model = Model(inputs, outputs)

In [ ]:
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.Tversky(
        alpha=0.3,
        beta=0.7
    ),
    metrics=["accuracy"]
)

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=5,
    restore_best_weights=True
)

In [ ]:
history = model.fit(
    train_data,
    validation_data=valid_data,
    epochs=30,
    callbacks=[early_stop]
)

Epoch 1/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 455s 10s/step - accuracy: 0.5921 - loss: 0.7935 - val_accuracy: 0.9261 - val_loss: 0.5776
Epoch 2/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 312s 7s/step - accuracy: 0.9357 - loss: 0.5387 - val_accuracy: 0.9511 - val_loss: 0.5008
Epoch 3/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 311s 7s/step - accuracy: 0.9545 - loss: 0.4785 - val_accuracy: 0.9592 - val_loss: 0.4654
Epoch 4/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 312s 7s/step - accuracy: 0.9521 - loss: 0.4892 - val_accuracy: 0.9632 - val_loss: 0.4610
Epoch 5/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 321s 7s/step - accuracy: 0.9544 - loss: 0.4712 - val_accuracy: 0.9492 - val_loss: 0.4916
Epoch 6/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 311s 7s/step - accuracy: 0.9547 - loss: 0.4866 - val_accuracy: 0.9633 - val_loss: 0.4568
Epoch 7/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 292s 7s/step - accuracy: 0.9567 - loss: 0.4533 - val_accuracy: 0.9657 - val_loss: 0.4545
Epoch 8/30
44/44 ━━━━━━━━━━━━━━━━━━━━ 309s 7s/step - accuracy: 0.9570 - loss: 0.4596 - val_accuracy: 0.9668 - 

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_data
)

print("Test Accuracy:", test_accuracy)

In [ ]:
import matplotlib.pyplot as plt

images, masks = next(iter(test_data))

prediction = model.predict(
    images[0:1],
    verbose=0
)[0]

prediction = (
    prediction > 0.5
).astype("uint8")

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.imshow(images[0])
plt.title("Original Image")
plt.axis("off")

plt.subplot(1, 3, 2)
plt.imshow(
    masks[0].numpy().squeeze(),
    cmap="gray"
)
plt.title("Actual Mask")
plt.axis("off")

plt.subplot(1, 3, 3)
plt.imshow(
    prediction.squeeze(),
    cmap="gray"
)
plt.title("Predicted Mask")
plt.axis("off")

plt.show()

In [ ]:
model.save("/content/drive/MyDrive/landslide_unet_final.keras")